# Limpieza de datos (EPH): de `eph_tecnologia_microdatos.csv` a `df_clean.csv`
**Práctica Profesional I - Rita Josefina Juan Briones**

En este cuaderno se **limpia** la base: se tratan los datos faltantes y los códigos de "sin respuesta", se recodifican las variables y se guarda el archivo final `df_clean.csv`. Cada decisión está explicada.

Ejecutá las celdas **en orden** (▶ o Shift + Enter). El archivo `eph_tecnologia_microdatos.csv` tiene que estar subido en Colab.

## Paso 1: Cargar los datos originales
Guardamos la base original en `df_raw` y **no la tocamos más**: todo el trabajo se hace sobre una copia.

In [1]:
import pandas as pd
import numpy as np

try:
    df_raw = pd.read_csv("eph_tecnologia_microdatos.csv")
    print("Archivo cargado:", df_raw.shape[0], "filas y", df_raw.shape[1], "columnas")
except FileNotFoundError:
    print("No encuentro el archivo. Subilo desde la carpeta de la izquierda y volve a ejecutar.")

Archivo cargado: 1000 filas y 8 columnas


## Paso 2: Ver qué hay que limpiar
Repasamos los dos problemas: los datos faltantes (`NaN`) y los códigos **9 = sin respuesta**.

In [2]:
print("Datos faltantes (NaN) por columna:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])

print("\nCodigos 9 (sin respuesta):")
print((df_raw[["NIVEL_ED", "IV12_3", "II7"]] == 9).sum())

Datos faltantes (NaN) por columna:
ITF       53
IV12_3    36
dtype: int64

Codigos 9 (sin respuesta):
NIVEL_ED    17
IV12_3      12
II7         15
dtype: int64


## Paso 3: Convertir los 9 en datos faltantes
El 9 significa "sin respuesta": no es información real. Si lo dejáramos, Python lo trataría como un número más y distorsionaría los resultados. Lo convertimos en `NaN` en una **copia** (`df`).

> La consigna menciona los códigos -9 y 99; en esta base el código de no respuesta es el **9**.

In [3]:
df = df_raw.copy()                       # trabajamos sobre una copia

for col in ["NIVEL_ED", "IV12_3", "II7"]:
    df[col] = df[col].replace(9, np.nan)  # 9 pasa a ser dato faltante

print("Datos faltantes despues de convertir los 9:")
print(df.isna().sum()[df.isna().sum() > 0])

Datos faltantes despues de convertir los 9:
NIVEL_ED    17
ITF         53
IV12_3      48
II7         15
dtype: int64


## Paso 4: Buscar valores imposibles (inconsistencias)
Controlamos que no haya ingresos negativos o en cero, hogares sin integrantes ni códigos de región que no existan.

In [4]:
print("ITF <= 0:", (df["ITF"] <= 0).sum())
print("IX_TOT < 1:", (df["IX_TOT"] < 1).sum())
print("Regiones validas:", df["REGION"].isin([1, 40, 41, 42, 43, 44]).all())
print("Deciles entre 1 y 10:", df["DECIFR"].between(1, 10).all())
print("CODUSU repetidos:", df["CODUSU"].duplicated().sum())

ITF <= 0: 0
IX_TOT < 1: 0
Regiones validas: True
Deciles entre 1 y 10: True
CODUSU repetidos: 0


## Paso 5: Decidir qué hacer con los datos faltantes
Se pueden **eliminar** las filas o **imputar** (inventar un valor). Decidí así:

- **Variables de resultado** (`IV12_3` y `II7`, tener PC e internet): si falta alguna, el hogar **no sirve** para probar las hipótesis → se **eliminan** esas filas.
- **Ingreso (`ITF`) y nivel educativo (`NIVEL_ED`)**: **no se imputan** (no quiero inventar ingresos ni estudios). Se dejan como `NaN` y cada análisis excluye solo los hogares que no tienen el dato que necesita. Así se pierde la menor cantidad posible de información.
- **Inconsistencia detectada:** los hogares sin `ITF` igual tienen decil (`DECIFR`) asignado. Se conserva el decil tal como viene en la base y se usa para los análisis por decil.

In [5]:
antes = len(df)
df = df.dropna(subset=["IV12_3", "II7"]).copy()
print("Filas antes:", antes, "| despues:", len(df), "| eliminadas:", antes - len(df))
print("\nFaltantes que se conservan:")
print(df.isna().sum()[df.isna().sum() > 0])

Filas antes: 1000 | despues: 937 | eliminadas: 63

Faltantes que se conservan:
NIVEL_ED    16
ITF         50
dtype: int64


## Paso 6: Recodificar las variables
Pasamos los códigos numéricos a etiquetas legibles y creamos variables 0/1 (más fáciles de calcular):
- `REGION_NOM` y `NIVEL_ED_NOM`: nombres en lugar de números.
- `TIENE_PC` y `TIENE_INTERNET`: 1 = sí, 0 = no.
- `GRUPO_INGRESO`: agrupa los deciles en bajo (1-3), medio (4-7) y alto (8-10).

In [6]:
df["REGION_NOM"] = df["REGION"].map({1: "GBA", 40: "NOA", 41: "NEA", 42: "Cuyo", 43: "Pampeana", 44: "Patagonia"})

df["NIVEL_ED_NOM"] = df["NIVEL_ED"].map({1: "Primaria incompleta", 2: "Primaria completa",
                                          3: "Secundaria incompleta", 4: "Secundaria completa",
                                          5: "Superior/Univ. completo"})

df["TIENE_PC"] = (df["IV12_3"] == 1).astype(int)          # 1 = si, 0 = no
df["TIENE_INTERNET"] = (df["II7"] == 1).astype(int)

df["GRUPO_INGRESO"] = pd.cut(df["DECIFR"], bins=[0, 3, 7, 10],
                             labels=["Bajo (deciles 1-3)", "Medio (deciles 4-7)", "Alto (deciles 8-10)"])

## Paso 7: Verificar el resultado
Quedamos solo con las columnas que se usan en el análisis y revisamos que todo esté bien.

In [7]:
columnas = ["CODUSU", "REGION", "REGION_NOM", "IX_TOT", "NIVEL_ED", "NIVEL_ED_NOM", "ITF",
            "DECIFR", "GRUPO_INGRESO", "TIENE_PC", "TIENE_INTERNET"]
df_clean = df[columnas]

print("Dimensiones:", df_clean.shape)
print("\nFaltantes:", df_clean.isna().sum()[df_clean.isna().sum() > 0].to_dict())
print("\nHogares por region:")
print(df_clean["REGION_NOM"].value_counts())
df_clean.head()

Dimensiones: (937, 11)

Faltantes: {'NIVEL_ED': 16, 'NIVEL_ED_NOM': 16, 'ITF': 50}

Hogares por region:
REGION_NOM
GBA          341
NEA          157
Pampeana     132
NOA          129
Patagonia     96
Cuyo          82
Name: count, dtype: int64


,CODUSU,REGION,REGION_NOM,IX_TOT,NIVEL_ED,NIVEL_ED_NOM,ITF,DECIFR,GRUPO_INGRESO,TIENE_PC,TIENE_INTERNET
0,HOG_EPH_2026_0001,40,NOA,3,3.0,Secundaria incompleta,198127.28,1,Bajo (deciles 1-3),1,1
1,HOG_EPH_2026_0002,44,Patagonia,6,4.0,Secundaria completa,NaN,6,Medio (deciles 4-7),1,0
2,HOG_EPH_2026_0003,42,Cuyo,5,4.0,Secundaria completa,1392730.43,9,Alto (deciles 8-10),1,1
3,HOG_EPH_2026_0004,41,NEA,4,4.0,Secundaria completa,420184.85,3,Bajo (deciles 1-3),0,0
4,HOG_EPH_2026_0005,1,GBA,2,2.0,Primaria completa,108507.59,1,Bajo (deciles 1-3),0,1


## Paso 8: Guardar `df_clean.csv`
Este es el archivo final que usa el análisis. `index=False` evita que se guarde una columna extra con los números de fila.

In [8]:
df_clean.to_csv("df_clean.csv", index=False)
print("Archivo guardado: df_clean.csv")

# Si estas en Colab, descarga el archivo a tu computadora (para subirlo a la carpeta /data de GitHub)
try:
    from google.colab import files
    files.download("df_clean.csv")
except ImportError:
    pass

Archivo guardado: df_clean.csv


## Resumen de la limpieza
| | Antes | Después |
|---|---|---|
| Hogares (filas) | 1.000 | **937** (se eliminaron 63 sin dato de PC o internet) |
| Códigos 9 | 17 + 12 + 15 | convertidos en datos faltantes |
| Faltantes que se conservan | | `ITF` 50 y `NIVEL_ED` 16 |
| Columnas | 8 | 11 (se agregan nombres, variables 0/1 y grupo de ingreso) |